# Add aligned H&E to DegaFiles and view it in Landscape

This notebook prepares a small Xenium example, adds an aligned H&E image to its
DegaFiles, and displays an interactive Celldega Landscape. The blue **FLUORESCENCE / H&E**
text control switches the background while cells and transcripts keep their coordinates.

We use the [SpatialData Xenium tutorial](https://spatialdata.scverse.org/en/stable/tutorials/notebooks/notebooks/examples/technology_xenium.html)'s
2,000 × 500 pixel region from the [10x Genomics FFPE Human Lung Cancer dataset](https://www.10xgenomics.com/datasets/preview-data-ffpe-human-lung-cancer-with-xenium-multimodal-cell-segmentation-1-standard)
(Xenium 2.0.0, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)).
The example contains 450 cells and 33,369 biological transcripts with QV ≥ 20.
Cells use a single display group; we do not infer cell types.

Run this notebook in local JupyterLab from a Celldega checkout. It reuses downloaded
files when present and creates a separate `data/xenium_he_notebook_dega` folder.
The first download is approximately 1.9 GB. The visualization server runs only on localhost.

## 1. Set up the environment and paths

Select the Python kernel with Celldega's preprocessing dependencies installed.
If needed, run these commands from the repository root before starting the notebook:

```bash
python -m pip install -e ".[pre]"
npm install
npm run build
```

libvips must be installed on the machine (for example, `brew install vips` on macOS).
Use the rebuilt frontend so the new image-source selector is available.

In [ ]:
import json
from pathlib import Path
import runpy

import numpy as np
import celldega as dega
from celldega import pre

# Works when Jupyter starts in the repository root or in notebooks/.
HERE = Path.cwd().resolve()
ROOT = next(
    path for path in (HERE, *HERE.parents)
    if (path / "scripts/prepare_xenium_he_example.py").is_file()
)
example_helpers = runpy.run_path(str(ROOT / "scripts/prepare_xenium_he_example.py"))
SAMPLE = example_helpers["SAMPLE"]
SOURCE_DIR = ROOT / "data/xenium_h_and_e"
DEGA_FILES = ROOT / "data/xenium_he_notebook_dega"
HE_IMAGE = SOURCE_DIR / f"{SAMPLE}_he_image.ome.tif"
ALIGNMENT_CSV = SOURCE_DIR / f"{SAMPLE}_he_imagealignment.csv"

print(f"Source data: {SOURCE_DIR}")
print(f"Output DegaFiles: {DEGA_FILES}")

## 2. Download or reuse the 10x example

The helper downloads the original H&E image and its alignment CSV, plus only the
DAPI, cell, transcript and expression files needed from the Xenium archive.
Large unused images and masks are skipped using HTTP byte-range requests.

In [ ]:
required = [SOURCE_DIR / name for name in example_helpers["FILES"]]
required += [HE_IMAGE, ALIGNMENT_CSV]
if any(not path.exists() for path in required):
    example_helpers["download"](SOURCE_DIR)
else:
    print("Reusing the downloaded 10x files.")

assert all(path.is_file() for path in required)

## 3. Prepare the fluorescence DegaFiles

For a quick example, reuse the helper's ordinary DAPI, cell and transcript preprocessing
for the tutorial region (`x=20000..22000`, `y=8000..8500` in original morphology pixels).
`include_he=False` leaves H&E for the next step, so its addition is explicit.

For your own dataset, run your normal `pre.main(...)` workflow first or start with
an existing DegaFiles folder. The H&E importer does not require rebuilding cells,
transcripts, clustering or gene-expression files.

In [ ]:
manifest_path = DEGA_FILES / "landscape_parameters.json"
if not DEGA_FILES.exists():
    example_helpers["prepare"](SOURCE_DIR, DEGA_FILES, include_he=False)
elif not manifest_path.is_file() or not (DEGA_FILES / "example.json").is_file():
    raise RuntimeError("Incomplete example output: choose a new DEGA_FILES directory.")
else:
    print("Reusing this notebook's existing DegaFiles.")

example = json.loads((DEGA_FILES / "example.json").read_text())
print(f"{example['cell_count']} cells; {example['transcript_count']:,} transcripts")
print("H&E already added:", "aligned_he" in json.loads(manifest_path.read_text()))

## 4. Add the aligned H&E image and save it to DegaFiles

The 10x CSV maps H&E pixels into **original morphology-image pixels** using column
vectors: `[x_morphology, y_morphology, 1] = M @ [x_he, y_he, 1]`.
Our DegaFiles use a crop, so compose the CSV with a translation into that crop.
There is no extra micron conversion for this CSV.

`pre.add_aligned_he()` resamples H&E onto the reference image grid, creates its
RGB tile pyramid, and records `aligned_he` in `landscape_parameters.json`. It also
supports DegaFiles stored in Parquet row groups. Existing fluorescence and spatial
files are preserved. It refuses to overwrite an existing H&E pyramid.

The supplied affine alignment can have local residual differences due to
post-staining tissue deformation. This step applies an existing alignment; it
does not estimate landmarks or perform registration.

In [ ]:
he_to_morphology = np.loadtxt(ALIGNMENT_CSV, delimiter=",")
x0, y0, width, height = example["crop_pixels"]
morphology_to_crop = np.array([
    [1, 0, -x0],
    [0, 1, -y0],
    [0, 0, 1],
], dtype=float)
he_to_dega = morphology_to_crop @ he_to_morphology

manifest_before = json.loads(manifest_path.read_text())
if "aligned_he" not in manifest_before:
    pre.add_aligned_he(DEGA_FILES, HE_IMAGE, alignment=he_to_dega)
else:
    print("H&E is already saved; skipping import on this rerun.")

manifest = json.loads(manifest_path.read_text())
assert manifest["image_info"] == manifest_before["image_info"]
assert (DEGA_FILES / "pyramid_images/h_and_e.dzi").is_file()
print(json.dumps(manifest["aligned_he"], indent=2))

### Use an existing DegaFiles folder

With an unmodified full Xenium morphology grid, pass the 10x CSV directly:

```python
pre.add_aligned_he(
    "path/to/DegaFiles",
    "sample_he_image.ome.tif",
    alignment="sample_he_imagealignment.csv",
)
```

If your H&E image is already resampled to the exact reference image grid, omit
`alignment`:

```python
pre.add_aligned_he("path/to/DegaFiles", "already_aligned_he.tif")
```

Without a matrix, image width, height, pixel size, origin and orientation must all
match. Equal dimensions alone do not establish alignment. The input must be an
8-bit RGB/RGBA image; matrices must map H&E pixels to DegaFiles image pixels.
The importer writes its output immediately; no separate save call is required.

## 5. Serve the saved DegaFiles locally

The widget loads image and spatial tiles over HTTP. This small server uses
Celldega's CORS handler, serves only the output folder, and chooses an available
localhost port. Leave the kernel running while viewing the Landscape.
For a notebook running on a remote machine, use a browser-accessible URL for the
same DegaFiles instead of this local-server cell.

In [ ]:
from functools import partial
from http.server import ThreadingHTTPServer
from threading import Thread

from celldega.viz.local_server import CORSHTTPRequestHandler

# Stop the previous instance if this cell is rerun.
if "he_server" in globals():
    he_server.shutdown()
    he_server.server_close()

handler = partial(CORSHTTPRequestHandler, directory=str(DEGA_FILES))
he_server = ThreadingHTTPServer(("127.0.0.1", 0), handler)
Thread(target=he_server.serve_forever, daemon=True).start()
base_url = f"http://127.0.0.1:{he_server.server_port}"
print(f"DegaFiles URL: {base_url}")

## 6. Display a Landscape and toggle the background

Run the cell below. In the image controls, click the blue **FLUORESCENCE** text and
choose **H&E**. Select **FLUORESCENCE** to return to DAPI. The menu appears because
the DegaFiles manifest advertises an aligned H&E image; no extra widget option is needed.

- **IMG** controls background visibility for either source.
- **CELL** and **TRX** control the spatial overlays.
- Zoom in to inspect cell boundaries and transcripts; search for **EPCAM** to highlight that gene.
- Fluorescence channel controls reappear when switching back from H&E.

If you add H&E after creating a widget, rerun this cell to load the updated manifest.

In [ ]:
landscape = dega.viz.Landscape(
    technology="Xenium",
    base_url=base_url,
    dataset_name="Human lung cancer - aligned H&E",
    ini_x=width / 2,
    ini_y=height / 2,
    ini_zoom=-1,
    height=600,
)
landscape

The saved DegaFiles are reusable in a new notebook or can be hosted together at
a new `base_url`. Yearbook reads the same manifest and offers the same image-source control.

When finished, optionally stop the local server (this leaves all DegaFiles on disk):

```python
he_server.shutdown()
he_server.server_close()
```

See [the aligned H&E guide](../docs/python/aligned_he.md) for the file format and
[the preparation script](../scripts/prepare_xenium_he_example.py) for the ROI preprocessing details.